# CH13 — one socket, many accelerators, no reboot

CH12 shipped nine bitstreams across four projects, and moving between them meant
**rebooting the board**. Not because of the bitstream: removing a device-tree
overlay leaks its `__symbols__` entries, so a later overlay declaring the same
`axi_iic` node is rejected with `EINVAL`, and every CH12 camera overlay declares
that node.

Dynamic Function eXchange removes the cause rather than the symptom. One overlay
is applied at boot and never removed; only a **partial** bitstream changes, and
`pr_download` applies no device tree at all.

This notebook loads the static design once, then swaps the filter in the socket
while the camera keeps streaming.

**What this is not.** A swap is not invisible — it takes hundreds of
milliseconds, which is tens of frames. The pipeline stalls and resumes with
different hardware in the socket. That is a great deal better than a reboot, and
it is worth being accurate about rather than selling as seamlessness.

In [ ]:
import pathlib, sys, time

for _cand in ("../../sw", "../sw", "sw", "."):
    if (pathlib.Path(_cand) / "dfx_socket.py").exists():
        sys.path.insert(0, str(pathlib.Path(_cand).resolve()))
        break
else:
    raise FileNotFoundError("cannot find dfx_socket.py -- copy sw/*.py next to this notebook")

print("sw/ ->", sys.path[0])

In [ ]:
import numpy as np
from pynq import Overlay, allocate, MMIO, Bitstream

# CH13 reuses CH12's camera driver unchanged -- the camera is in the STATIC
# region, which is the whole reason it survives a swap. Import before the
# Overlay is constructed: PYNQ resolves hierarchy drivers newest-registered
# first, and pynq.lib.video has already registered Pcam5C.
for candidate in ("../../sw", "../sw", "../../../CH12/sw", "."):
    if (pathlib.Path(candidate) / "ov5647.py").exists():
        sys.path.insert(0, candidate)
        break
import ov5647 as cam

import dfx_socket as dfx
import rm_ref as ref

W, H = 1280, 720
OUT = pathlib.Path("../out")

ol = Overlay(str(OUT / "dfx_socket.bit"))
mipi = ol.mipi
mipi.pipeline.release_video_reset()
m = mipi.configure(mode="1280x720")
mipi.start()
time.sleep(0.5)
mipi.auto_white_balance()
print(f"camera: {m.width}x{m.height}, chip {mipi.sensor.chip_id():#06x}")

## Ask the static region first, always

This is the rule the whole chapter is built around, and it is not a style
preference.

On ZynqMP there is **no bus timeout on the PL ports**. A read of a partition
that is held in reset, mid-reconfiguration, or empty does not return an error —
it does not return at all. The CPU stops, with no panic and no console output,
and only a power cycle recovers it. "Held in reset", "being reconfigured" and
"the logic is broken" are also indistinguishable from outside.

So the status GPIO lives in the **static** region. Static logic always answers;
the partition may not. Every access below goes through it.

In [ ]:
# A Block Design Container is exposed by PYNQ as a HIERARCHY, not as an IP:
# `ol.socket` is a DefaultHierarchy with no read()/write(), and the .hwh carries
# no register map for it either, because a container does not propagate its
# inner IP's. MMIO addresses it directly, by the address the build assigned --
# which is what the driver wants anyway, since it works in raw offsets
# precisely so a swap cannot invalidate the metadata underneath it.
# ol.dfx_ctrl IS an ordinary IP, so it needs no such treatment.
socket_ip = MMIO(0xB0000000, 0x10000)

socket = dfx.DfxSocket(socket_ip, ol.dfx_ctrl)

# The partition comes out of reset held, so release it before asking anything.
socket.release_reset()
socket.release_shutdown()

print("heartbeat toggling :", socket.alive())
print("held in reset      :", socket.in_reset())
print("status word        :", f"{socket.status():#06x}")
print("kernel in socket   :", socket.kernel_name())

## The swap

Seven steps, in an order where each one exists because something went wrong
without it:

```
1. wait for ap_idle            the accelerator must not be mid-frame
2. engage the shutdown managers  and WAIT until all three acknowledge
3. hold the partition in reset
4. pr_download(partial)        no dtbo -- this is what avoids the reboot
5. release the partition reset
6. check the heartbeat         is the new RM actually alive?
7. release the managers        only now may traffic reach the socket
   then read kernel_id         and confirm what is really in there
```

Steps 2 and 7 are what the spike's kernel panic bought: reconfiguring
underneath live AXI traffic gave
`Kernel panic - not syncing: Asynchronous SError Interrupt`. Step 6 is what the
board wedges bought.

In [ ]:
def swap_to(name):
    """Swap the socket to one RM and report how long it took."""
    partial = OUT / f"socket_{name}_partial.bit"
    if not partial.exists():
        cands = sorted(p.name for p in OUT.glob("*partial*.bit"))
        raise FileNotFoundError(f"{partial.name} not found -- have: {cands}")

    t0 = time.time()
    got = socket.swap(lambda: Bitstream(str(partial), partial=True).download(),
                      getattr(ref, f"KERNEL_{name.upper()}"))
    dt = time.time() - t0

    steps = socket.trace
    print(f"swapped to {name} in {dt*1e3:.0f} ms, kernel_id = {got:#010x}")
    for (a, ta), (_, tb) in zip(steps, steps[1:] + [(None, time.time())]):
        print(f"    {a:<18} {(tb-ta)*1e3:7.1f} ms")
    return dt

swap_to("blur")

## Bit-exact, per kernel

The socket contract does not change across a swap — same register map, same
addresses, same driver. What changes is **what the hardware computes**, and the
`mode` register's meaning along with it: a menu selection for the sobel RM, a
threshold level for the threshold RM, nothing at all for the blur.

That is exactly why `kernel_id` exists. You cannot read `mode` correctly
without knowing which kernel is in the socket, and the only trustworthy way to
know is to ask the hardware.

In [ ]:
src = allocate(shape=(H, W, 4), dtype=np.uint8)
dst = allocate(shape=(H, W, 4), dtype=np.uint8)

mipi.readframe()                      # discard; the first after start can be partial
shot = mipi.readframe()
src[:] = np.asarray(shot)             # freeze it, so the VDMA cannot write into it
src.flush()
frozen = np.array(src)

def run_and_check(kernel, mode):
    socket.check_ready()
    ip = socket_ip
    ip.write(dfx.REG_MODE, mode)
    ip.write(0x10, src.device_address & 0xFFFFFFFF)
    ip.write(0x14, src.device_address >> 32)
    ip.write(0x1C, dst.device_address & 0xFFFFFFFF)
    ip.write(0x20, dst.device_address >> 32)
    ip.write(0x28, W)
    ip.write(0x30, H)
    t0 = time.time()
    ip.write(dfx.REG_CTRL, dfx.AP_START)
    while not (ip.read(dfx.REG_CTRL) & dfx.AP_DONE):
        pass
    dt = time.time() - t0
    dst.invalidate()
    exp = ref.filter_frame(frozen, kernel, mode)
    diff = np.abs(np.array(dst)[:, :, :3].astype(np.int16) - exp[:, :, :3].astype(np.int16))
    print(f"  mode {mode:<4} {dt*1e3:6.2f} ms   differing {int(np.count_nonzero(diff)):7d}"
          f"   max {int(diff.max()):3d}   {'ok' if diff.max() == 0 else 'MISMATCH'}")

print("blur:")
run_and_check(ref.KERNEL_BLUR, 0)

In [ ]:
swap_to("threshold")
print("threshold, the mode register is the LEVEL:")
for level in (64, 128, 200):
    run_and_check(ref.KERNEL_THRESHOLD, level)

In [ ]:
swap_to("sobel")
print("sobel, the mode register is a MENU:")
for mode in ref.SOBEL_MODES:
    run_and_check(ref.KERNEL_SOBEL, mode)

## What a swap costs

The number that matters, and the one to be honest about. The partial bitstream's
size is set by the **partition**, not by what is in it: the partition is a whole
clock region because `RESET_AFTER_RECONFIG` requires clock-region alignment, and
that region holds roughly five times the largest RM.

So the reconfiguration time below is a property of how the socket was sized, not
of the filter. A partition fitted to the accelerator would swap proportionally
faster — and that trade is itself something this chapter can show.

In [ ]:
sizes = {p.name: p.stat().st_size for p in sorted(OUT.glob("*.bit"))}
for n, s in sizes.items():
    print(f"  {n:<52} {s/1024:8.0f} KB")

print()
times = {}
for name in ("passthrough", "blur", "threshold", "sobel"):
    times[name] = swap_to(name)
print()
print(f"median swap: {sorted(times.values())[len(times)//2]*1e3:.0f} ms"
      f"  ({sorted(times.values())[len(times)//2] * 60:.0f} frames at 60 fps)")

## Clean up

In [ ]:
mipi.close()
src.freebuffer()
dst.freebuffer()